Sofia Anaya Palafox

738594

INGENIERÍA FINANCIERA

## Pregunta 1

In [9]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize

def f(x):
    x1, x2, x3 = x
    t1 = ((6*x1-2)**2) * np.sin(12*x1-4)
    t2 = ((4*x2-3)**2) * np.cos(9*x2-1)
    t3 = ((5*x3-1)**2) * np.sin(8*x3-4)
    return t1 + t2 + t3

np.random.seed(42)
n_iniciales = 5
n_iteraciones = 15

# Muestras iniciales aleatorias
X = np.random.rand(n_iniciales, 3)
y = np.array([f(x) for x in X])

print(f"Muestras iniciales: {n_iniciales}")
print(f"Mejor valor inicial: {np.min(y):.6f}\n")

# Bucle
for i in range(n_iteraciones):
    #  Entrenar Gaussian Process
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)

    # Encontrar el punto con máxima incertidumbre (desviación estándar más alta)
    def neg_std(x):
        _, sigma = gp.predict(x.reshape(1, -1), return_std=True)
        return -sigma[0]  # Negativo porque minimizamos

    best_x = None
    best_neg_std = -np.inf

    for _ in range(10):  # Probar 10 puntos iniciales aleatorios
        x0 = np.random.rand(3)
        res = minimize(neg_std, x0, bounds=[(0,1)]*3, method='L-BFGS-B')
        if -res.fun > best_neg_std:
            best_neg_std = -res.fun
            best_x = res.x

    x_next = best_x

    #  Evaluar la función real en el punto de máxima incertidumbre
    y_next = f(x_next)

    # 4. Actualizar datos
    X = np.vstack([X, x_next])
    y = np.append(y, y_next)

    print(f"Iteración {i+1:2d}: ({x_next[0]:.3f}, {x_next[1]:.3f}, {x_next[2]:.3f}) → f={y_next:.6f} | Mejor={np.min(y):.6f}")

# Resultados
mejor_idx = np.argmin(y)
print(f"\n--- Resultado ---")
print(f"Mejor valor: {y[mejor_idx]:.6f}")
print(f"Mejor punto: x={X[mejor_idx][0]:.4f}, y={X[mejor_idx][1]:.4f}, z={X[mejor_idx][2]:.4f}")

Muestras iniciales: 5
Mejor valor inicial: -7.680714

Iteración  1: (1.000, 1.000, 0.000) → f=16.441034 | Mejor=-7.680714
Iteración  2: (0.000, 0.000, 0.666) → f=13.167333 | Mejor=-7.680714
Iteración  3: (1.000, 0.739, 1.000) → f=3.722479 | Mejor=-7.680714
Iteración  4: (0.000, 0.506, 0.000) → f=2.911725 | Mejor=-7.680714
Iteración  5: (0.361, 1.000, 0.000) → f=0.620596 | Mejor=-7.680714
Iteración  6: (0.210, 0.436, 1.000) → f=-14.189292 | Mejor=-14.189292
Iteración  7: (0.819, 0.711, 0.457) → f=-4.263312 | Mejor=-14.189292
Iteración  8: (0.295, 0.419, 0.474) → f=-2.047427 | Mejor=-14.189292
Iteración  9: (1.000, 0.283, 0.670) → f=21.300221 | Mejor=-14.189292
Iteración 10: (0.476, 0.569, 0.023) → f=0.917182 | Mejor=-14.189292
Iteración 11: (0.000, 0.000, 0.000) → f=8.646733 | Mejor=-14.189292
Iteración 12: (1.000, 0.564, 0.000) → f=16.254707 | Mejor=-14.189292
Iteración 13: (0.000, 1.000, 1.000) → f=-9.227130 | Mejor=-14.189292
Iteración 14: (0.479, 0.000, 0.574) → f=7.582041 | Mejor=-

**Explica por qué optimización bayesiana es una elección buena para este problema en lugar de GridSearch.**

Eficiencia: El Grid Search prueba todas las combinaciones de una tabla, una por una. Si hay muchas opciones, se vuelve lentísimo. En cambio, la optimización bayesiana va paso a paso y elige inteligentemente el siguiente punto más prometedor, probando muchas menos veces.

Exploración vs. Explotación: La optimización bayesiana ya equilibra explorar zonas inciertas y explotar zonas buenas. El Grid Search no sabe hacer eso.

Sin rejilla: Con Grid Search tienes que definir valores y pasos fijos. Si los pasos son grandes, te saltas el óptimo. Si son pequeños, explota el número de combinaciones. La optimización bayesiana se mueve de forma continua y flexible.

## Pregunta 2

In [10]:
data = pd.read_csv('adidas.csv')
data = data.drop(columns=['url', 'name', 'sku', 'description', 'images', 'source_website', 'source', 'breadcrumbs', 'language', 'currency', 'color', 'crawled_at'])
data['original_price'] = data['original_price'].str.replace('$', '').astype(float)
data['average_rating'] = data['average_rating'].apply(lambda x: 1 if x >= 4.3 else 0)
num_col = ['selling_price', 'original_price', 'reviews_count']
cat_col = ['availability', 'category', 'brand', 'country']
data = data.dropna().reset_index(drop=True)
data

,selling_price,original_price,availability,category,brand,country,average_rating,reviews_count
0,20,25.0,InStock,Clothing,adidas,USA,1,116
1,20,25.0,InStock,Clothing,adidas,USA,1,116
2,20,25.0,InStock,Clothing,adidas,USA,1,116
3,48,80.0,InStock,Clothing,adidas,USA,0,144
4,64,80.0,InStock,Shoes,adidas,USA,1,160
...,...,...,...,...,...,...,...,...
824,72,120.0,InStock,Shoes,adidas,USA,1,151
825,70,100.0,InStock,Shoes,adidas,USA,1,135
826,35,50.0,InStock,Shoes,adidas,USA,1,190
827,40,50.0,InStock,Shoes,adidas,USA,1,190


In [11]:
X = data.drop(columns=['average_rating'])
X = pd.get_dummies(X, columns=cat_col, drop_first=True)
y = data['average_rating']
x_test, x_train, y_test, y_train = train_test_split(X, y, train_size=0.7, random_state=42)

In [12]:
x_test

,selling_price,original_price,reviews_count,availability_OutOfStock,category_Clothing,category_Shoes
145,18,30.0,53,False,True,False
306,32,35.0,1352,False,True,False
234,24,30.0,78,False,True,False
220,28,35.0,7291,False,False,True
819,44,55.0,177,False,False,True
...,...,...,...,...,...,...
71,48,60.0,214,False,True,False
106,28,35.0,3812,False,False,True
270,52,65.0,671,False,True,False
435,44,55.0,17,False,False,True


In [13]:
scaler = StandardScaler().fit(x_train[num_col])
x_train[num_col] = scaler.transform(x_train[num_col])
x_test[num_col] = scaler.transform(x_test[num_col])

In [14]:
regresion = LogisticRegression()
regresion.fit(x_train,y_train)
y_pred = regresion.predict_proba(x_train)[:, 1]
roc_auc = roc_auc_score(y_train, y_pred)
roc_auc

np.float64(0.7683724509426701)

In [15]:
y_pred_test = regresion.predict_proba(x_test)[:, 1]
roc_auc_test = roc_auc_score(y_test, y_pred_test)
roc_auc_test

np.float64(0.6790495049504951)

In [16]:
n = x_train.shape[0]
p = x_train.shape[1]
n,p

(249, 6)

In [17]:
n = x_train.shape[0]
p = x_train.shape[1]

RSS = np.sum((y_pred_test - y_test) ** 2)
var = RSS / (n - p - 1)

b_0 = np.ravel(regresion.intercept_)
b_i = np.ravel(regresion.coef_)

X = np.column_stack((np.ones(n), x_train))
X = X.astype(float)

var_beta = np.linalg.inv(X.T @ X + np.eye(X.shape[1]) * 1e-6) * var
std_beta = np.sqrt(np.diag(var_beta))

betas = np.concatenate((b_0, b_i))

t_stats = betas / std_beta
p_values = [2 * (1 - stats.t.cdf(np.abs(t), n - p - 1)) for t in t_stats]

for i, (b, t, pval) in enumerate(zip(betas, t_stats, p_values)):
    if i == 0:
        print(f"Intercepto: = {b:.4f}, estadístico t = {t:.4f}, p-value = {pval:.4e}")

        if pval < 0.05:
            print("-> El intercepto es estadísticamente significativo.")
        else:
            print("-> El intercepto no es estadísticamente significativo.")
    else:
        print(f"beta {i}: {b:.4f}, estadístico t = {t:.4f}, p-value = {pval:.4e}")

        if pval < 0.05:
            print(f"-> El coeficiente beta {i} es significativo.")
        else:
            print(f"-> El coeficiente beta {i} no es significativo.")

Intercepto: = 3.0323, estadístico t = 27.5615, p-value = 0.0000e+00
-> El intercepto es estadísticamente significativo.
beta 1: -0.5052, estadístico t = -2.9998, p-value = 2.9838e-03
-> El coeficiente beta 1 es estadísticamente significativo.
beta 2: -0.2007, estadístico t = -1.1718, p-value = 2.4243e-01
-> El coeficiente beta 2 no es estadísticamente significativo.
beta 3: 0.8751, estadístico t = 25.8792, p-value = 0.0000e+00
-> El coeficiente beta 3 es estadísticamente significativo.
beta 4: 0.0000, estadístico t = 0.0000, p-value = 1.0000e+00
-> El coeficiente beta 4 no es estadísticamente significativo.
beta 5: -0.3411, estadístico t = -2.9060, p-value = 3.9996e-03
-> El coeficiente beta 5 es estadísticamente significativo.
beta 6: -0.5403, estadístico t = -4.2038, p-value = 3.6966e-05
-> El coeficiente beta 6 es estadísticamente significativo.


**Explica el significado del score que arroje tu modelo. Analiza el resultado de los p-values encontrados.**

Las variables selling_price, reviews_count, category_Clothing y category_Shoes son estadísticamente significativas para predecir la average_rating. Por otro lado, original_price y availability_OutOfStock no muestran un impacto significativo en el modelo

## Pregunta 3

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

df_diabetes = pd.read_csv('diabetes.csv')

display(df_diabetes.head())
df_diabetes.info()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [2]:
missing_value_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Remplazar ceros
for col in missing_value_cols:
    df_diabetes[col] = df_diabetes[col].replace(0, np.nan)

# Separar "X" y "y"
X = df_diabetes.drop('Outcome', axis=1)
y = df_diabetes['Outcome']

# training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train value counts:\n{y_train.value_counts(normalize=True)}")
print(f"y_test value counts:\n{y_test.value_counts(normalize=True)}")

X_train shape: (537, 8)
X_test shape: (231, 8)
y_train value counts:
Outcome
0    0.651769
1    0.348231
Name: proportion, dtype: float64
y_test value counts:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [6]:
# Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

# GridSearchCV
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2']
}

# StratifiedKFold para cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=kf, scoring='f1', n_jobs=-1, verbose=2)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_f1_cv = grid_search.best_score_

print(f"\Mejores hiperparametros: {best_params}")
print(f" F1-score  (CV): {best_f1_cv:.4f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits


<>:27: SyntaxWarning: invalid escape sequence '\M'
<>:27: SyntaxWarning: invalid escape sequence '\M'
/tmp/ipykernel_17578/2941062988.py:27: SyntaxWarning: invalid escape sequence '\M'
  print(f"\Mejores hiperparametros: {best_params}")


\Mejores hiperparametros: {'classifier__C': 0.1, 'classifier__penalty': 'l1'}
 F1-score  (CV): 0.6563


In [4]:
# Predicciones para el mejor modelo
y_pred_test = best_model.predict(X_test)

# Calacular F1-score test
f1_test = f1_score(y_test, y_pred_test)

print(f"F1-score test {f1_test:.4f}")
print("\nComparasion:")
print(f"Promedio F1-score en training: {best_f1_cv:.4f}")
print(f"F1-score en test : {f1_test:.4f}")


print("\nClasificacion Test:")
print(classification_report(y_test, y_pred_test))

if abs(best_f1_cv - f1_test) < 0.05:
    print("\nEl modelo está generalizando bien, ya que el F1-score en el conjunto de prueba es cercano al promedio de CV.")
else:
    print("\nEl modelo podría no estar generalizando tan bien, ya que hay una diferencia notable entre el F1-score de CV y el F1-score del conjunto de prueba.")

F1-score on the test set: 0.5655

Comparison:
Average F1-score on training set (CV): 0.6563
F1-score on the test set: 0.5655

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.76      0.85      0.80       150
           1       0.64      0.51      0.57        81

    accuracy                           0.73       231
   macro avg       0.70      0.68      0.68       231
weighted avg       0.72      0.73      0.72       231


El modelo podría no estar generalizando tan bien, ya que hay una diferencia notable entre el F1-score de CV y el F1-score del conjunto de prueba.


El modelo funciona un poco peor con datos nuevos que con los que usamos para validarlo por dentro. Eso quiere decir que puede que el modelo esté un poco "memorizado" (sobreajustado) o que no esté captando del todo bien cómo son los datos reales de prueba. Tal vez habría que ajustarlo más o conseguir más datos para entrenarlo.